# Comparing Unfitted Methods on a Common Geometry

This notebook is a tutorial-style comparison of unfitted finite element methods applied to the same embedded boundary-value problem.
The main idea is to keep the geometry, mesh, and PDE fixed, and then change only the discretization strategy.

This makes the comparison meaningful: differences in behavior can be attributed to the unfitted method itself rather than to a change in the model problem.

The notebook is organized in two parts:

1. A shared setup: background mesh, embedded geometry, problem data, and continuous model.
2. Separate method blocks that discretize that same model in different ways.

This structure is intended to support direct comparison across multiple unfitted formulations while keeping the underlying problem fixed.


In [ ]:
using Gridap 
using GridapEmbedded

## 1. Model Problem

Let $\Omega \subset \mathbb{R}^3$ denote the embedded domain labeled `csg`, and let $\Gamma_D$ denote the internal boundary labeled `source`.
The notebook considers the Poisson problem

$$
-\Delta u = f \quad \text{in } \Omega,
$$

with Dirichlet boundary condition

$$
u = u_D \quad \text{on } \Gamma_D,
$$

where in this example $f=10$ and $u_D=1$ are constants.

In strong form, this is a scalar elliptic boundary-value problem posed on a geometry that is not fitted by the background Cartesian mesh.
The purpose of the notebook is to approximate this same continuous problem with multiple unfitted methods, so the model stated here is the common reference for every method block below.

The corresponding weak form is: find $u \in V_D$ such that

$$
\int_{\Omega} \nabla u \cdot \nabla v\,\mathrm{d}x = \int_{\Omega} f v\,\mathrm{d}x
$$

for all test functions $v \in V_0$, where

$$
V_D = \{ w \in H^1(\Omega) : w = u_D \text{ on } \Gamma_D \},
\qquad
V_0 = \{ w \in H^1(\Omega) : w = 0 \text{ on } \Gamma_D \}.
$$

Since the mesh does not conform to $\Gamma_D$, the Dirichlet condition is imposed weakly in the discrete formulations through Nitsche-type terms.
The methods differ in how they obtain a stable unfitted discretization of this same weak problem: CutFEM adds ghost-penalty stabilization on cut-cell skeletons, whereas AgFEM modifies the discrete space through cell aggregation.


## 2. Background Mesh

All methods considered in this notebook start from the same unfitted background mesh.
We build a uniform Cartesian mesh on a box, and the actual computational domain will later be extracted from it by cutting with an embedded geometry.

The parameter `n` controls the number of cells in each coordinate direction. Changing `n` therefore refines the common discretization baseline for every method in the comparison.

In [ ]:
# Background Mesh
n = 10
partition = (n,n,n)
pmin = 0.8*Point(-1,-1,-1)
pmax = 0.8*Point(1,1,1)
bgmodel = CartesianDiscreteModel(pmin,pmax,partition)

## 3. Geometry Definition

The embedded geometry is created with constructive solid geometry.
Three cylinders aligned with the coordinate axes are united into a shape named `source`.
That shape is then removed from the intersection of a sphere and a cube to define the physical domain `csg`.

The result is a non-trivial embedded domain whose boundary is not aligned with the background mesh.
This is precisely the regime where unfitted methods are useful, and it gives a common geometric benchmark for comparing how different formulations handle cut cells and embedded boundaries.

In [ ]:
# Defining the geometry 
R = 0.5
geo1 = cylinder(R,v=VectorValue(1,0,0))
geo2 = cylinder(R,v=VectorValue(0,1,0))
geo3 = cylinder(R,v=VectorValue(0,0,1))
geo4 = union(union(geo1,geo2),geo3,name="source")
geo5 = sphere(1)
geo6 = cube(L=1.5)
geo7 = intersect(geo6,geo5)
geo8 = setdiff(geo7,geo4,name="csg")

## 4. Problem Data

We prescribe a constant source term `f` in the bulk domain and a constant Dirichlet value `ud` on the embedded boundary.
These simple data are deliberate: they keep the notebook focused on the comparison of unfitted discretizations, rather than on complications coming from variable coefficients or elaborate boundary data.

In [ ]:
# Forcing data
ud = 1
f = 10

## 5. Method 1: CutFEM

This is the first method block in the comparison.
It keeps the shared model problem unchanged and introduces the CutFEM discretization on the common background mesh.

First, the background mesh is cut with the geometry `csg`. From that cut representation we extract:

- `Ω`: the physical domain where the PDE is solved,
- `Γd`: the embedded boundary where Dirichlet conditions are imposed,
- `Γg`: the ghost skeleton used for ghost-penalty stabilization.

The Dirichlet condition is imposed weakly with a Nitsche-type term controlled by `γd`.
The method-specific ingredient is the ghost-penalty term controlled by `γg`, which stabilizes the discretization across cut-cell interfaces.

After defining the measures, finite element spaces, bilinear form, and linear form, we assemble the affine operator and solve for `uh`.

In the notebook comparison, CutFEM serves as the formulation that stabilizes the problem by adding extra terms to the weak form.

In [ ]:
# Cut Fem implementation 

# Cut the background model
cutgeo = cut(bgmodel,geo8)

# Setup integration meshes
Ω = Triangulation(cutgeo,PHYSICAL,"csg")
Γd = EmbeddedBoundary(cutgeo,"csg","source")
Γg = GhostSkeleton(cutgeo,"csg")

# Setup normal vectors
n_Γd = get_normal_vector(Γd)
n_Γg = get_normal_vector(Γg)

# Setup Lebesgue measures
order = 1
degree = 2*order
dΩ = Measure(Ω,degree)
dΓd = Measure(Γd,degree)
dΓg = Measure(Γg,degree)

# Setup FESpace
Ω_act = Triangulation(cutgeo,ACTIVE,"csg")
V = TestFESpace(Ω_act,ReferenceFE(lagrangian,Float64,order),conformity=:H1)
U = TrialFESpace(V)

# Weak form
γd = 10.0
γg = 0.1
h = (pmax - pmin)[1] / partition[1]

a(u,v) =
∫( ∇(v)⋅∇(u) ) * dΩ +
∫( (γd/h)*v*u  - v*(n_Γd⋅∇(u)) - (n_Γd⋅∇(v))*u ) * dΓd +
∫( (γg*h)*jump(n_Γg⋅∇(v))*jump(n_Γg⋅∇(u)) ) * dΓg

l(v) =
∫( v*f ) * dΩ +
∫( (γd/h)*v*ud - (n_Γd⋅∇(v))*ud ) * dΓd

# FE problem
op = AffineFEOperator(a,l,U,V)
uh = solve(op)

# Post processing
writevtk(Ω,"postprocess/trian_O")
writevtk(Γd,"postprocess/trian_Gd",cellfields=["normal"=>n_Γd])
writevtk(Γg,"postprocess/trian_Gg")
writevtk(Triangulation(bgmodel),"postprocess/bgtrian")
writevtk(Ω,"postprocess/cutfem_solution",cellfields=["uh"=>uh])

## 6. Method 2: AgFEM

This section implements the aggregated finite element method (AgFEM) for exactly the same geometry, mesh, and Poisson problem used in the CutFEM block.
The comparison is therefore method-to-method: the continuous problem stays fixed, while the discrete stabilization mechanism changes.

The key AgFEM idea is to stabilize the unfitted discretization by constraining degrees of freedom associated with problematic cut cells through aggregates.
Unlike CutFEM, AgFEM does not introduce a ghost-penalty term on a skeleton. Instead, stability is built into the discrete space by replacing the standard active FE space with an aggregated one.

For tutorial purposes, the code below also exports aggregate information on the background mesh. This makes it possible to inspect how the aggregation pattern differs from the CutFEM treatment of small cut cells.


In [ ]:
# AgFEM implementation

# Cut the background model
cutgeo = cut(bgmodel,geo8)

# Setup integration meshes
Ω_ag = Triangulation(cutgeo,PHYSICAL,"csg")
Ω_ag_act = Triangulation(cutgeo,ACTIVE,"csg")
Γd_ag = EmbeddedBoundary(cutgeo,"csg","source")

# Setup normal vectors
n_Γd_ag = get_normal_vector(Γd_ag)

# Setup Lebesgue measures
order_ag = 1
degree_ag = 2*order_ag
dΩ_ag = Measure(Ω_ag,degree_ag)
dΓd_ag = Measure(Γd_ag,degree_ag)

# Setup the standard active FE space
reffe_ag = ReferenceFE(lagrangian,Float64,order_ag)
Vstd_ag = TestFESpace(Ω_ag_act,reffe_ag,conformity=:H1)

# Aggregate cut cells following the AgFEM strategy
aggregates = aggregate(AggregateCutCellsByThreshold(1.0),cutgeo,geo8,IN)
V_ag = AgFEMSpace(Vstd_ag,aggregates)
U_ag = TrialFESpace(V_ag)

# Weak form
γd_ag = 10.0
h_ag = (pmax - pmin)[1] / partition[1]

a_ag(u,v) =
∫( ∇(v)⋅∇(u) ) * dΩ_ag +
∫( (γd_ag/h_ag)*v*u - v*(n_Γd_ag⋅∇(u)) - (n_Γd_ag⋅∇(v))*u ) * dΓd_ag

l_ag(v) =
∫( v*f ) * dΩ_ag +
∫( (γd_ag/h_ag)*v*ud - (n_Γd_ag⋅∇(v))*ud ) * dΓd_ag

# FE problem
op_ag = AffineFEOperator(a_ag,l_ag,U_ag,V_ag)
uh_ag = solve(op_ag)

# Post processing
colors_ag = color_aggregates(aggregates,bgmodel)
writevtk(Triangulation(bgmodel),"postprocess/agfem_aggregates",celldata=["aggregate"=>aggregates,"color"=>colors_ag])
writevtk(Ω_ag,"postprocess/agfem_solution",cellfields=["uh"=>uh_ag])

## 7. Comparison Notes and Post-processing

The `writevtk` calls export method-specific geometrical and solution data that can be inspected in ParaView.
These files make it possible to compare not only the computed solutions, but also the auxiliary structures introduced by each unfitted method, such as ghost skeletons or aggregates.

Because every method reuses the same mesh size, geometry, source term, and boundary data, the notebook supports direct side-by-side comparison.
Useful comparison points include accuracy, conditioning, method-specific stabilization, ease of implementation, and sensitivity to small cut cells.